# Build Working Dataset

This notebook takes our raw NYC taxi data (the fhvhv parquet files we downloaded, about 30 GB) and turns it into a smaller, cleaner working dataset that we will use later for our PySpark project.

What this notebook does:
1. Look inside the raw data folder and find every parquet file.
2. Open each file, one at a time and keep only the columns we aare interested in.
3. Remove rows that have missing or impossible values (for example, a trip with a negative fare).
4. Save the cleaned version of each file into a new "working" folder
5. Print a summary showing how many rows we kept and how big our new working dataset is.

## Import the tools

In [1]:
import os
import pandas as pd

## Desired Columns and FIle Paths

In [2]:
# Folder containing raw parquet files
RAW_DATA_FOLDER = "C:/Users/thiga/nyc_tlc_raw"

# Folder to save our new working dataset
WORKING_DATA_FOLDER = "C:/Users/thiga/OneDrive/Documents/Masters - Big Data/Year 1/Semester 2/MIT 805/nyc_tlc_working"

# The columns we want to keep for our project.
# (NYC TLC changed the format as the years passed. Therefore, not every column exists in every file)
COLUMNS_WE_WANT = [
    "hvfhs_license_num",
    "pickup_datetime",
    "dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_miles",
    "trip_time",
    "base_passenger_fare",
    "tips",
    "driver_pay",
    "shared_request_flag",]

## Find our raw files

This function looks inside a folder and gives us back a list of just the parquet file names (it ignores anything that is not a parquet file).

In [3]:
def find_raw_files(folder_path):
    file_names = os.listdir(folder_path)

    parquet_file_names = []
    for name in file_names:
        if name.endswith(".parquet"):
            parquet_file_names.append(name)

    return parquet_file_names

In [4]:
# Check the size of our RAW dataset (the original downloaded files, before any cleaning has been done)
raw_file_list_for_size_check = find_raw_files(RAW_DATA_FOLDER)

total_raw_size_in_bytes = 0
for file_name in raw_file_list_for_size_check:
    file_path = os.path.join(RAW_DATA_FOLDER, file_name)
    total_raw_size_in_bytes = total_raw_size_in_bytes + os.path.getsize(file_path)

total_raw_size_in_gb = total_raw_size_in_bytes / (1024 ** 3)

print("Number of raw files: " + str(len(raw_file_list_for_size_check)))
print("Total raw dataset size: " + str(round(total_raw_size_in_gb, 2)) + " GB")

Number of raw files: 73
Total raw dataset size: 30.21 GB


## Check for schema drift across years

NYC TLC changed the file format multiple times over the years - added new columns, and even changed how some existing columns are stored. Let's check three specific columns (that a teammate's analysis needs) across three different years, so we can see this happening for ourselves instead of just assuming it.

In [5]:
# Look at three columns that changed over the years, in three different years
#
# Note: we do NOT use pd.read_parquet() here. That would load the entire file
# into memory (every column, every row) just to check 3 column names, which
# can run out of memory on a laptop. Instead we use pyarrow to just peek at
# the file's schema (its column names and types), which is almost instant
# and barely uses any memory at all.
import pyarrow.parquet as pq

files_to_check = [
    "fhvhv_tripdata_2019-02.parquet",
    "fhvhv_tripdata_2022-01.parquet",
    "fhvhv_tripdata_2025-05.parquet",
]

columns_to_check = ["airport_fee", "cbd_congestion_fee", "wav_match_flag"]

for file_name in files_to_check:
    file_path = os.path.join(RAW_DATA_FOLDER, file_name)
    file_schema = pq.ParquetFile(file_path).schema_arrow

    print("File: " + file_name)
    for column_name in columns_to_check:
        if column_name in file_schema.names:
            column_type = file_schema.field(column_name).type
            print("  " + column_name + " - present, data type: " + str(column_type))
        else:
            print("  " + column_name + " - not present in this file")
    print("")

File: fhvhv_tripdata_2019-02.parquet
  airport_fee - present, data type: null
  cbd_congestion_fee - not present in this file
  wav_match_flag - present, data type: null

File: fhvhv_tripdata_2022-01.parquet
  airport_fee - present, data type: double
  cbd_congestion_fee - not present in this file
  wav_match_flag - present, data type: string

File: fhvhv_tripdata_2025-05.parquet
  airport_fee - present, data type: double
  cbd_congestion_fee - present, data type: double
  wav_match_flag - present, data type: large_string



## Clean one month of data

This function reads one raw parquet file and cleans it:
1. Keeps only the columns we want
2. Removes rows with missing values in the important columns
3. Removes rows with values that don't make sense (like a negative fare)


In [6]:
def clean_one_month(file_path):
    # Read the raw file into a table
    data = pd.read_parquet(file_path)

    # Work out which of our wanted columns actually exist in this file (older files may be missing some columns)
    columns_available = []
    for column_name in COLUMNS_WE_WANT:
        if column_name in data.columns:
            columns_available.append(column_name)

    data = data[columns_available]

    # Remove rows with missing or impossible values, but only for columns that are actually present in this file
    if "trip_miles" in data.columns:
        data = data.dropna(subset=["trip_miles"])
        data = data[data["trip_miles"] > 0]

    if "trip_time" in data.columns:
        data = data.dropna(subset=["trip_time"])
        data = data[data["trip_time"] > 0]

    if "base_passenger_fare" in data.columns:
        data = data.dropna(subset=["base_passenger_fare"])
        data = data[data["base_passenger_fare"] >= 0]

    return data

## Test if the above works on one file

In [7]:
raw_file_list = find_raw_files(RAW_DATA_FOLDER)
print("Found " + str(len(raw_file_list)) + " raw files.")

# Try cleaning just the first file
first_file_path = os.path.join(RAW_DATA_FOLDER, raw_file_list[0])
test_result = clean_one_month(first_file_path)

test_result.head()

Found 73 raw files.


,hvfhs_license_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,base_passenger_fare,tips,driver_pay,shared_request_flag
0,HV0003,2019-02-01 00:05:18,2019-02-01 00:14:57,245,251,2.45,579,9.35,0.0,7.48,Y
1,HV0003,2019-02-01 00:41:29,2019-02-01 00:49:39,216,197,1.71,490,7.91,2.0,7.93,N
2,HV0005,2019-02-01 00:51:34,2019-02-01 01:28:29,261,234,5.01,2159,44.96,0.0,35.97,N
3,HV0005,2019-02-01 00:03:51,2019-02-01 00:07:16,87,87,0.34,179,7.19,3.0,5.39,N
4,HV0005,2019-02-01 00:09:44,2019-02-01 00:39:56,87,198,6.84,1799,24.25,4.0,17.07,N


## Process every raw file

In [8]:
# Check if the folder we are saving to exists
if not os.path.exists(WORKING_DATA_FOLDER):
    os.makedirs(WORKING_DATA_FOLDER)

# Used to keep track of progress
total_rows_before_cleaning = 0
total_rows_after_cleaning = 0
files_done_so_far = 0
files_skipped_because_already_done = 0

# Go through every raw file, one at a time
for file_name in raw_file_list:
    files_done_so_far = files_done_so_far + 1

    raw_file_path = os.path.join(RAW_DATA_FOLDER, file_name)
    working_file_path = os.path.join(WORKING_DATA_FOLDER, file_name)

    # If a file has already been processed, don't redo it.
    if os.path.exists(working_file_path):
        files_skipped_because_already_done = files_skipped_because_already_done + 1

        # We still want an accurate "rows after cleaning" total, so we
        # read the row count from the working file we already saved
        already_cleaned_data = pd.read_parquet(working_file_path)
        total_rows_after_cleaning = total_rows_after_cleaning + len(already_cleaned_data)

        print("File " + str(files_done_so_far) + " of " + str(len(raw_file_list))
              + " (" + file_name + "): already done in an earlier run, skipping")
        continue

    # Read the raw file just to count its rows before cleaning
    raw_data = pd.read_parquet(raw_file_path)
    rows_before = len(raw_data)

    # Clean this file
    cleaned_data = clean_one_month(raw_file_path)
    rows_after = len(cleaned_data)

    # Save the cleaned version into our working folder, using the
    # same file name so it's easy to match it back to the raw file
    cleaned_data.to_parquet(working_file_path)

    # Update our running totals
    total_rows_before_cleaning = total_rows_before_cleaning + rows_before
    total_rows_after_cleaning = total_rows_after_cleaning + rows_after

    # Print progress so we can see it working
    print("File " + str(files_done_so_far) + " of " + str(len(raw_file_list)) + " (" + file_name + "): kept " + str(rows_after)
          + " rows out of " + str(rows_before))

File 1 of 73 (fhvhv_tripdata_2019-02.parquet): already done in an earlier run, skipping
File 2 of 73 (fhvhv_tripdata_2019-03.parquet): already done in an earlier run, skipping
File 3 of 73 (fhvhv_tripdata_2019-04.parquet): already done in an earlier run, skipping
File 4 of 73 (fhvhv_tripdata_2019-05.parquet): already done in an earlier run, skipping
File 5 of 73 (fhvhv_tripdata_2019-06.parquet): already done in an earlier run, skipping
File 6 of 73 (fhvhv_tripdata_2019-07.parquet): already done in an earlier run, skipping
File 7 of 73 (fhvhv_tripdata_2019-08.parquet): already done in an earlier run, skipping
File 8 of 73 (fhvhv_tripdata_2019-09.parquet): already done in an earlier run, skipping
File 9 of 73 (fhvhv_tripdata_2019-10.parquet): already done in an earlier run, skipping
File 10 of 73 (fhvhv_tripdata_2019-11.parquet): already done in an earlier run, skipping
File 11 of 73 (fhvhv_tripdata_2019-12.parquet): already done in an earlier run, skipping
File 12 of 73 (fhvhv_tripdata_

## Summary

Let's add up the size of everything in our working folder, and print a summary of what we did. This is the number you report as your **working dataset size** for the assignment.

In [8]:
# Work out the total size of the working folder
total_size_in_bytes = 0
working_file_list = find_raw_files(WORKING_DATA_FOLDER)
for file_name in working_file_list:
    file_path = os.path.join(WORKING_DATA_FOLDER, file_name)
    total_size_in_bytes = total_size_in_bytes + os.path.getsize(file_path)

total_size_in_gb = total_size_in_bytes / (1024 ** 3)

print("WORKING DATASET SUMMARY")
print("Files processed: " + str(files_done_so_far))
print("Total rows after cleaning: " + str(total_rows_after_cleaning))
print("Working dataset size: " + str(round(total_size_in_gb, 2)) + " GB")

WORKING DATASET SUMMARY
Files processed: 73
Total rows after cleaning: 1283180517
Working dataset size: 25.22 GB
